In [2]:
import json
from pathlib import Path
from collections import Counter

PROJECT_ROOT = Path.home() / "xai-knowledge-graph"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

INPUT_PATH = PROCESSED_DIR / "papers_enriched.json"
FINAL_PATH = PROCESSED_DIR / "papers_final.json"

with open(INPUT_PATH) as f:
    papers = json.load(f)
print(f"Loaded {len(papers)} enriched papers")

Loaded 3913 enriched papers


In [3]:
TOPIC_VOCAB = {
    "LIME":                  ["LIME", "local interpretable model-agnostic"],
    "SHAP":                  ["SHAP", "Shapley", "SHapley Additive"],
    "Grad-CAM":              ["Grad-CAM", "GradCAM", "gradient-weighted class activation"],
    "Saliency":              ["saliency map", "saliency-based", "pixel attribution"],
    "Attention":             ["attention map", "attention visualiz", "attention mechanism"],
    "Counterfactual":        ["counterfactual"],
    "Feature Attribution":   ["feature attribution", "feature importance"],
    "Integrated Gradients":  ["integrated gradient"],
    "Interpretability":      ["interpretability", "interpretable"],
    "Explainability":        ["explainability", "explainable AI", "XAI"],
    "Transparency":          ["transparency", "transparent model"],
    "Fairness":              ["fairness", "fair ML", "algorithmic fairness"],
    "Bias":                  ["bias mitigation", "debiasing", "bias detection"],
    "Trust":                 ["trustworthy", "trust in AI"],
    "Causal":                ["causal inference", "causality"],
    "Post-hoc":              ["post-hoc", "post hoc"],
    "Model-Agnostic":        ["model-agnostic", "model agnostic"],
    "Concept Activation":    ["concept activation", "TCAV"],
    "Rule-based":            ["rule-based explanation", "decision rule"],
    "Prototype":             ["prototype-based", "prototype example"],
    "Influence Function":    ["influence function"],
    "Adversarial":           ["adversarial example", "adversarial robust"],
    "Computer Vision":       ["computer vision", "image classification", "convolutional neural"],
    "NLP":                   ["natural language processing", "language model", "text classif"],
    "Tabular":               ["tabular data", "structured data"],
    "Time Series":           ["time series", "temporal data"],
    "Healthcare":            ["healthcare", "medical", "clinical", "diagnosis"],
    "Reinforcement":         ["reinforcement learning"],
    "Graph Neural Networks": ["graph neural", "GNN ", "graph attention"],
    "Federated":             ["federated learning"],
}
print(f"Vocabulary: {len(TOPIC_VOCAB)} topics")

Vocabulary: 30 topics


In [4]:
def extract_topics(text):
    tl = text.lower()
    matched = set()
    for topic, terms in TOPIC_VOCAB.items():
        for term in terms:
            if term.lower() in tl:
                matched.add(topic)
                break
    return sorted(matched)

for p in papers:
    p["topics"] = extract_topics(f"{p['title']} {p['abstract']}")

zero_topics = sum(1 for p in papers if not p["topics"])
print(f"Papers with 0 topics: {zero_topics} ({zero_topics/len(papers)*100:.1f}%)")
print(f"Avg topics per paper: {sum(len(p['topics']) for p in papers) / len(papers):.2f}")

Papers with 0 topics: 45 (1.2%)
Avg topics per paper: 2.92


In [5]:
with open(FINAL_PATH, "w", encoding="utf-8") as f:
    json.dump(papers, f, indent=2, ensure_ascii=False)

# Also save flat reference lookups for Day 2's Neo4j load
authors_set = sorted({a for p in papers for a in p["authors"]})
with open(PROCESSED_DIR / "authors.json", "w") as f:
    json.dump(authors_set, f, indent=2)
with open(PROCESSED_DIR / "topics.json", "w") as f:
    json.dump(list(TOPIC_VOCAB.keys()), f, indent=2)

# Stats
topic_counts = Counter()
for p in papers:
    for t in p["topics"]:
        topic_counts[t] += 1


print(f"  Papers:  {len(papers):>5} → {FINAL_PATH.name}")
print(f"  Authors: {len(authors_set):>5} → authors.json")
print(f"  Topics:  {len(TOPIC_VOCAB):>5} → topics.json")
print(f"\nTopic distribution:")
for topic, c in topic_counts.most_common():
    bar = "█" * int(c / max(topic_counts.values()) * 30)
    print(f"  {c:>5}  {topic:<22} {bar}")

  Papers:   3913 → papers_final.json
  Authors: 13754 → authors.json
  Topics:     30 → topics.json

Topic distribution:
   2246  Explainability         ██████████████████████████████
   1818  Interpretability       ████████████████████████
   1057  SHAP                   ██████████████
    903  Healthcare             ████████████
    583  Transparency           ███████
    483  NLP                    ██████
    476  Feature Attribution    ██████
    443  Computer Vision        █████
    405  Counterfactual         █████
    396  Grad-CAM               █████
    364  LIME                   ████
    331  Saliency               ████
    298  Post-hoc               ███
    273  Trust                  ███
    232  Model-Agnostic         ███
    168  Integrated Gradients   ██
    153  Attention              ██
    151  Time Series            ██
    121  Fairness               █
    116  Tabular                █
     89  Graph Neural Networks  █
     88  Reinforcement          █
     55  Cau